# discriminator-classifier-head — worked example 1: Scalar real/fake head with einops rearrange instead of flatten

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `discriminator-classifier-head`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The end of a DCGAN discriminator collapses a `(B, C, H, W)` feature map to one probability per image. The classic recipe is `Rearrange('b c h w -> b (c h w)')`, then `Linear(C*H*W, 1, bias=False)`, then `Sigmoid()`. Using einops `rearrange` makes the flatten explicit and order-preserving, and the bias-free `Linear` is just a single dot product of each flattened image against the weight row.

## Worked solution

**Goal:** map `features: (B, C, H, W)` to `probs: (B,)`.

1. **Flatten everything but the batch axis.** `rearrange(features, 'b c h w -> b (c h w)')` produces `(B, C*H*W)`. einops walks the source axes left-to-right, so the flattened layout matches what `features.flatten(start_dim=1)` would give — this matters because `weight` was trained against that exact ordering.
2. **Apply the bias-free Linear.** `weight` has shape `(1, C*H*W)`. The affine map is `flat @ weight.T`, giving `(B, 1)`. There is no bias term here (the atom's definition uses `bias=False`), so we do not add anything.
3. **Sigmoid to a probability.** `t.sigmoid(logits)` squashes each logit into `(0, 1)`.
4. **Squeeze the singleton output axis.** `.squeeze(-1)` turns `(B, 1)` into `(B,)`, one scalar probability per image.

Why this works: a discriminator only needs to answer one yes/no question per image ("is this real?"). Flattening discards spatial structure deliberately — by this depth the conv stack has already aggregated location-specific evidence into channels, so a single linear projection to one logit is enough.

In [ ]:
def disc_head_rearrange(features: Tensor, weight: Tensor) -> Tensor:
    flat = rearrange(features, 'b c h w -> b (c h w)')   # (B, C*H*W)
    logits = flat @ weight.T                             # (B, 1), bias-free
    return t.sigmoid(logits).squeeze(-1)                 # (B,)

t.manual_seed(0)
B, C, H, W = 4, 8, 4, 4
features = t.randn(B, C, H, W)
weight = t.randn(1, C * H * W)
probs = disc_head_rearrange(features, weight)
print('shape:', tuple(probs.shape))
print('all in (0,1):', bool(((probs > 0) & (probs < 1)).all()))
print('probs:', probs.round(decimals=3).tolist())